In [1]:
import pandas as pd
import sqlite3

# Загружаем данные
file_path = r"C:\Users\chamo\Downloads\dashboard_data.xlsx"
df = pd.read_excel(file_path)

# Создаём подключение к SQLite (база в памяти)
conn = sqlite3.connect(":memory:")

# Сохраняем DataFrame как таблицу в SQLite
df.to_sql("loans", conn, if_exists="replace", index=False)

print("✅ Таблица 'loans' создана в SQLite!")
print(f"   Колонки: {list(df.columns)}")
print(f"   Всего строк: {len(df)}")

# Проверяем: выполняем простой SQL-запрос
query = "SELECT * FROM loans LIMIT 3;"
result = pd.read_sql_query(query, conn)
print("\nРезультат запроса (первые 3 строки):")
print(result)

✅ Таблица 'loans' создана в SQLite!
   Колонки: ['Площадка', 'вид_кредита', 'Бакет', 'Решение', 'Дата подписания документов', 'Время на решение (часы)', 'Неделя регистрации', 'Месяц регистрации', 'Одобрено', 'Подписано', 'номер заявки']
   Всего строк: 11031

Результат запроса (первые 3 строки):
  Площадка вид_кредита Бакет   Решение Дата подписания документов  \
0   Восток          ИК     0  Отказать                       None   
1   Восток          КК  1-30  Отказать                       None   
2   Восток          ПК     0  Отказать                       None   

   Время на решение (часы)   Неделя регистрации    Месяц регистрации  \
0               473.742500  2020-12-21 00:00:00  2020-12-01 00:00:00   
1               237.594722  2021-01-11 00:00:00  2021-01-01 00:00:00   
2                80.373056  2021-01-11 00:00:00  2021-01-01 00:00:00   

   Одобрено  Подписано  номер заявки  
0         0          0             1  
1         0          0             2  
2         0         

In [2]:
# ==========================================
# SQL-ЗАПРОСЫ ПО ДАННЫМ О РЕСТРУКТУРИЗАЦИИ
# ==========================================

# 1. Общая статистика
query_1 = """
SELECT 
    COUNT(*) as total_loans,
    AVG("Время на решение (часы)") as avg_ttd_hours,
    AVG(Одобрено) * 100 as approval_rate_percent,
    AVG(Подписано) * 100 as conversion_rate_percent
FROM loans;
"""
result_1 = pd.read_sql_query(query_1, conn)
print("=== ОБЩАЯ СТАТИСТИКА ===")
print(result_1.round(2))

# 2. Среднее время по площадкам
query_2 = """
SELECT 
    Площадка,
    COUNT(*) as total_loans,
    AVG("Время на решение (часы)") as avg_ttd_hours
FROM loans
GROUP BY Площадка
ORDER BY avg_ttd_hours DESC;
"""
result_2 = pd.read_sql_query(query_2, conn)
print("\n=== СРЕДНЕЕ ВРЕМЯ ПО ПЛОЩАДКАМ ===")
print(result_2.round(2))

# 3. Уровень одобрения по бакетам
query_3 = """
SELECT 
    Бакет,
    COUNT(*) as total_loans,
    AVG(Одобрено) * 100 as approval_rate_percent
FROM loans
GROUP BY Бакет
ORDER BY approval_rate_percent DESC;
"""
result_3 = pd.read_sql_query(query_3, conn)
print("\n=== УРОВЕНЬ ОДОБРЕНИЯ ПО БАКЕТАМ ===")
print(result_3.round(2))

# 4. Входящий поток по месяцам
query_4 = """
SELECT 
    "Месяц регистрации" as month,
    COUNT(*) as total_loans
FROM loans
GROUP BY "Месяц регистрации"
ORDER BY month;
"""
result_4 = pd.read_sql_query(query_4, conn)
print("\n=== ВХОДЯЩИЙ ПОТОК ПО МЕСЯЦАМ ===")
print(result_4)

=== ОБЩАЯ СТАТИСТИКА ===
   total_loans  avg_ttd_hours  approval_rate_percent  conversion_rate_percent
0        11031          100.6                  21.04                    16.67

=== СРЕДНЕЕ ВРЕМЯ ПО ПЛОЩАДКАМ ===
  Площадка  total_loans  avg_ttd_hours
0   Восток         2906         106.91
1    Запад         8125          98.35

=== УРОВЕНЬ ОДОБРЕНИЯ ПО БАКЕТАМ ===
    Бакет  total_loans  approval_rate_percent
0       0         6660                  24.52
1    1-30         1827                  18.88
2   31-90         1481                  16.68
3  91-180          564                  12.06
4    181+          499                   5.61

=== ВХОДЯЩИЙ ПОТОК ПО МЕСЯЦАМ ===
                 month  total_loans
0  2020-09-01 00:00:00           14
1  2020-10-01 00:00:00            5
2  2020-11-01 00:00:00           10
3  2020-12-01 00:00:00          624
4  2021-01-01 00:00:00         8312
5  2021-02-01 00:00:00         2066


In [3]:
SELECT 
    "Месяц регистрации" as month,
    COUNT(*) as total_loans,
    AVG(COUNT(*)) OVER (ORDER BY "Месяц регистрации" ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING) as moving_avg
FROM loans
GROUP BY "Месяц регистрации"
ORDER BY month;

IndentationError: unexpected indent (3204951911.py, line 2)

In [4]:
SELECT 
    "Месяц_регистрации" as month,
    COUNT(*) as total_loans,
    AVG(COUNT(*)) OVER (ORDER BY "Месяц_регистрации" ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING) as moving_avg
FROM loans
GROUP BY "Месяц_регистрации"
ORDER BY month;

IndentationError: unexpected indent (1572747945.py, line 2)

In [5]:
query_5 = """
SELECT 
    "Месяц регистрации" as month,
    COUNT(*) as total_loans,
    AVG(COUNT(*)) OVER (ORDER BY "Месяц регистрации" ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING) as moving_avg
FROM loans
GROUP BY "Месяц регистрации"
ORDER BY month;
"""

result_5 = pd.read_sql_query(query_5, conn)
print("=== СКОЛЬЗЯЩЕЕ СРЕДНЕЕ ПО МЕСЯЦАМ ===")
print(result_5)

=== СКОЛЬЗЯЩЕЕ СРЕДНЕЕ ПО МЕСЯЦАМ ===
                 month  total_loans   moving_avg
0  2020-09-01 00:00:00           14     9.500000
1  2020-10-01 00:00:00            5     9.666667
2  2020-11-01 00:00:00           10   213.000000
3  2020-12-01 00:00:00          624  2982.000000
4  2021-01-01 00:00:00         8312  3667.333333
5  2021-02-01 00:00:00         2066  5189.000000


-- ==========================================
-- АНАЛИЗ ДАННЫХ О РЕСТРУКТУРИЗАЦИИ КРЕДИТОВ
-- ==========================================

-- 1. Общая статистика
SELECT 
    COUNT(*) as total_loans,
    AVG("Время на решение (часы)") as avg_ttd_hours,
    AVG(Одобрено) * 100 as approval_rate_percent,
    AVG(Подписано) * 100 as conversion_rate_percent
FROM loans;

-- 2. Среднее время по площадкам
SELECT 
    Площадка,
    COUNT(*) as total_loans,
    AVG("Время на решение (часы)") as avg_ttd_hours
FROM loans
GROUP BY Площадка
ORDER BY avg_ttd_hours DESC;

-- 3. Уровень одобрения по бакетам
SELECT 
    Бакет,
    COUNT(*) as total_loans,
    AVG(Одобрено) * 100 as approval_rate_percent
FROM loans
GROUP BY Бакет
ORDER BY approval_rate_percent DESC;

-- 4. Входящий поток по месяцам
SELECT 
    "Месяц регистрации" as month,
    COUNT(*) as total_loans
FROM loans
GROUP BY "Месяц регистрации"
ORDER BY month;

-- 5. Скользящее среднее по месяцам (оконная функция)
SELECT 
    "Месяц регистрации" as month,
    COUNT(*) as total_loans,
    AVG(COUNT(*)) OVER (ORDER BY "Месяц регистрации" ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING) as moving_avg
FROM loans
GROUP BY "Месяц регистрации"
ORDER BY month;